In [3]:
!python3 -m pip install torch
!python3 -m pip install sktime
!python3 -m pip install git+https://github.com/gon-uri/detach_rocket

import os
import random
import time
import h5py
import numpy as np
import pandas as pd
import scipy.io
import librosa
import kagglehub
import IPython.display as ipd
from IPython.display import display, Audio
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Nueva importación para Detach-ROCKET
from detach_rocket.detach_classes import DetachRocket

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Defaulting to user installation because normal site-packages is not writeable
  Cloning https://github.com/gon-uri/detach_rocket to /private/var/folders/q0/g33km23d63774hrjz3lxmh2h0000gn/T/pip-req-build-mvi7ndym
  Running command git clone -q https://github.com/gon-uri/detach_rocket /private/var/folders/q0/g33km23d63774hrjz3lxmh2h0000gn/T/pip-req-build-mvi7ndym
  Resolved https://github.com/gon-uri/detach_rocket to commit aa046a32c7244f61ea21bdda9cd21708f8663274
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade p

In [4]:
# ==================================================
# 2. CARGA Y CONCATENACIÓN DE BABBLE NOISE 
# ==================================================
babble_root_path = kagglehub.dataset_download("artharking/babble-noise-mgdcc")

audio_buffers_list = []
max_archivos_a_combinar = 40 
sr_original = 16000
sr_objetivo = 22050

archivos_mat = 0
archivos_wav = 0

for root, dirs, files in os.walk(babble_root_path):
    for f in files:
        file_path = os.path.join(root, f)
        
        if f.lower().endswith('.mat'):
            archivos_mat += 1
            try:
                mat_contents = scipy.io.loadmat(file_path)
                keys = [k for k in mat_contents.keys() if not k.startswith('__')]
                if keys:
                    audio_raw = mat_contents[keys[0]].flatten().astype(np.float32)
                    if len(audio_raw) > 0: audio_buffers_list.append(audio_raw)
            except:
                try:
                    with h5py.File(file_path, 'r') as f_h5:
                        keys = list(f_h5.keys())
                        if keys:
                            audio_raw = np.array(f_h5[keys[0]]).flatten().astype(np.float32)
                            if len(audio_raw) > 0: audio_buffers_list.append(audio_raw)
                except:
                    pass
                    
        elif f.lower().endswith('.wav'):
            archivos_wav += 1
            try:
                y, _ = librosa.load(file_path, sr=sr_original)
                if len(y) > 0: audio_buffers_list.append(y)
            except:
                pass

        if len(audio_buffers_list) >= max_archivos_a_combinar:
            break
            
    if len(audio_buffers_list) >= max_archivos_a_combinar:
        break

print("-" * 50)
print(f"Estadísticas de escaneo profundo:")
print(f" - Archivos .mat encontrados: {archivos_mat}")
print(f" - Archivos .wav encontrados: {archivos_wav}")
print(f" - Archivos extraídos con éxito: {len(audio_buffers_list)}")

if len(audio_buffers_list) > 0:
    print("\nConcatenando y remuestreando a 22050 Hz...")
    babble_completo_16k = np.concatenate(audio_buffers_list)
    babble_audio_full = librosa.resample(babble_completo_16k, orig_sr=sr_original, target_sr=sr_objetivo)

    print(f"Babble noise estructurado correctamente")
    print(f"Duración total del murmullo: {len(babble_audio_full)/sr_objetivo:.2f} segundos.")
else:
    print("\nNo hay audios válidos en el dataset.")
    babble_audio_full = None

--------------------------------------------------
Estadísticas de escaneo profundo:
 - Archivos .mat encontrados: 40
 - Archivos .wav encontrados: 0
 - Archivos extraídos con éxito: 40

Concatenando y remuestreando a 22050 Hz...
Babble noise estructurado correctamente
Duración total del murmullo: 29.64 segundos.


In [5]:

# ==================================================
# 3. FUNCIONES DE DATA AUGMENTATION
# ==================================================
def add_white_noise(audio, noise_level=0.005):
    noise = np.random.randn(len(audio))
    return audio + noise_level * noise

def add_pink_noise(audio, noise_level=0.01):
    white = np.random.randn(len(audio))
    fft_white = np.fft.rfft(white)
    frequencies = np.maximum(np.fft.rfftfreq(len(audio)), 1e-10)
    f_filter = 1.0 / np.sqrt(frequencies)
    f_filter /= np.max(f_filter)
    fft_pink = fft_white * f_filter
    pink = np.fft.irfft(fft_pink, n=len(audio))
    pink = pink / np.max(np.abs(pink))
    return audio + noise_level * pink

def add_babble_noise(audio, babble_audio, noise_level=0.03):
    if babble_audio is None:
        return audio

    if len(babble_audio) < len(audio):
        babble_audio = np.tile(babble_audio, int(np.ceil(len(audio) / len(babble_audio))))

    start_idx = random.randint(0, len(babble_audio) - len(audio))
    babble_chunk = babble_audio[start_idx : start_idx + len(audio)]
    babble_chunk = babble_chunk / (np.max(np.abs(babble_chunk)) + 1e-10)

    return audio + noise_level * babble_chunk

In [6]:
# ==========================================
# 4. TRANSFORMACIÓN Y DATA AUGMENTATION
# ==========================================
correct_audio_dir = None

for root, dirs, files in os.walk(dataset_root_path):
    if any(f.endswith('.wav') for f in files):
        correct_audio_dir = root
        break

X_list = []
y_list = []

print(f"DataFrame listo: {len(df_filtrado)} audios base listos para ser multiplicados.")

for index, row in df_filtrado.iterrows():
    file_path = os.path.join(correct_audio_dir, row['filename'])
    categoria = row['category']

    try:
        y_audio, sr = librosa.load(file_path, sr=22050)

        audios_a_procesar = [
            y_audio,                                     
            add_white_noise(y_audio),                    
            add_pink_noise(y_audio),                     
            add_babble_noise(y_audio, babble_audio_full) 
        ]

        for audio_version in audios_a_procesar:
            mfccs = librosa.feature.mfcc(y=audio_version, sr=sr, n_mfcc=40)
            mfccs_scaled_features = np.mean(mfccs.T, axis=0)
            X_list.append(mfccs_scaled_features)
            y_list.append(categoria)

    except Exception as e:
        print(f"Error procesando {file_path}: {e}")

X = np.array(X_list)
y = np.array(y_list)

# Agregamos la dimensión de canal requerida
X = np.expand_dims(X, axis=1)

print("-" * 50)
print("Pipeline completado.")
print(f"Dimensiones de X finales: {X.shape}")
print(f"Total de audios procesados: {len(y)} (Originales x 4)")

DataFrame listo: 120 audios base listos para ser multiplicados.
--------------------------------------------------
Pipeline completado.
Dimensiones de X finales: (480, 1, 40)
Total de audios procesados: 480 (Originales x 4)


In [7]:
# ==========================================
# 5. ENTRENAMIENTO CON DETACH-ROCKET
# ==========================================
from sklearn.metrics import accuracy_score # Asegúrate de importar esto

le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

print("-" * 50)
print("RESUMEN DE AUDIOS UTILIZADOS:")
print(f"Total de muestras (con Data Augmentation): {len(X)}")
print(f" - Muestras para Entrenamiento: {len(X_train)}")
print(f" - Muestras para Prueba (Test): {len(X_test)}")
print("-" * 50)

inicio = time.time()

# Instanciar y ajustar Detach-ROCKET
detach_model = DetachRocket('rocket', num_kernels=5000)
detach_model.fit(X_train, y_train)

# Hacer predicciones sobre el conjunto de prueba
y_pred = detach_model.predict(X_test)

# Calcular la métrica con sklearn
accuracy = accuracy_score(y_test, y_pred)
fin = time.time()

print(f"Precisión (Test Accuracy): {accuracy * 100:.2f}%")
print(f"Tiempo total de ejecución: {fin - inicio:.2f} segundos")

# DetachRocket permite visualizar qué proporción de variables (features) retuvo:
if hasattr(detach_model, 'feature_proportion_'):
    print(f"Porcentaje de features retenidas tras el podado: {detach_model.feature_proportion_ * 100:.2f}%")

# ==========================================
# 6. AUDICIÓN DE VARIANTES
# ==========================================
if 'df_filtrado' in locals() and 'correct_audio_dir' in locals() and correct_audio_dir is not None:
    random_row = df_filtrado.sample(n=1).iloc[0]
    random_file_path = os.path.join(correct_audio_dir, random_row['filename'])
    random_label = random_row['category']

    print("=" * 50)
    print(f"AUDICIÓN DE VARIANTES: {random_label.upper()}")
    print(f"Archivo base: {random_row['filename']}")
    print("=" * 50)

    try:
        y_audio_test, sr_audio = librosa.load(random_file_path, sr=22050)
        
        print("\n1. Audio Original:")
        display(ipd.Audio(y_audio_test, rate=sr_audio))
        
        print("\n2. Variante: Ruido Blanco:")
        y_white = add_white_noise(y_audio_test)
        display(ipd.Audio(y_white, rate=sr_audio))
            
        print("\n3. Variante: Ruido Rosa:")
        y_pink = add_pink_noise(y_audio_test)
        display(ipd.Audio(y_pink, rate=sr_audio))
            
        if babble_audio_full is not None:
            print("\n4. Variante: Murmullo de fondo (Babble Noise):")
            y_babble = add_babble_noise(y_audio_test, babble_audio_full)
            display(ipd.Audio(y_babble, rate=sr_audio))
        else:
            print("\n[Aviso: No se generó Babble Noise válido]")
        
    except Exception as e:
        print(f"Error interno al intentar procesar los audios: {e}")

--------------------------------------------------
RESUMEN DE AUDIOS UTILIZADOS:
Total de muestras (con Data Augmentation): 480
 - Muestras para Entrenamiento: 384
 - Muestras para Prueba (Test): 96
--------------------------------------------------


/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/pytho

TRAINING RESULTS Full ROCKET:
Optimal Alpha Full ROCKET: 3.36
Train Accuraccy Full ROCKET: 100.00%
-------------------------


/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/pytho

TRAINING RESULTS Detach Model:
Optimal Alpha Detach Model: 3.36
Train Accuraccy Detach Model: 99.74%
-------------------------
Precisión (Test Accuracy): 97.92%
Tiempo total de ejecución: 0.93 segundos
AUDICIÓN DE VARIANTES: CAT
Archivo base: 4-133047-C-5.wav

1. Audio Original:


/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/rociocaseres/Library/Python/3.9/lib/python/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b



2. Variante: Ruido Blanco:



3. Variante: Ruido Rosa:



4. Variante: Murmullo de fondo (Babble Noise):
